In [0]:
spark.conf.set(
  "fs.azure.account.key.mypersonalfinance09.dfs.core.windows.net",
  "account key"
)

In [0]:
dbutils.fs.ls("abfss://finance@mypersonalfinance09.dfs.core.windows.net/raw")

In [0]:
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.fs.ls("abfss://finance@mypersonalfinance09.dfs.core.windows.net/raw/2026-02/")

In [0]:
import pandas as pd

path = "abfss://finance@mypersonalfinance09.dfs.core.windows.net/raw/2026-02/transactions.xlsx"
local_path = "/tmp/transactions.xlsx"

dbutils.fs.cp(path, "file:" + local_path)

pdf = pd.read_excel(local_path)

df = spark.createDataFrame(pdf)

display(df)

In [0]:
from pyspark.sql.functions import current_timestamp

df = df.withColumn("ingestion_timestamp", current_timestamp())

In [0]:
df.write.format("delta") \
.mode("overwrite") \
.save("abfss://finance@mypersonalfinance09.dfs.core.windows.net/bronze/bronze_transactions")

In [0]:
bronze_df = spark.read.format("delta").load("abfss://finance@mypersonalfinance09.dfs.core.windows.net/bronze/bronze_transactions")
display(bronze_df)


In [0]:
bronze_df.count()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window = Window.partitionBy("transaction_id") \
.orderBy(col("ingestion_timestamp").desc())

silver_df = bronze_df.withColumn(
    "rn", row_number().over(window)
).filter(
    col("rn") == 1
).drop("rn")

In [0]:
silver_df.count()
display(silver_df)

In [0]:
silver_path = "abfss://finance@mypersonalfinance09.dfs.core.windows.net/silver/silver_transactions"
silver_df.write.format("delta") \
.mode("overwrite") \
.save(silver_path)

In [0]:
display(silver_df)

In [0]:
%sql
     DESCRIBE TABLE delta.`abfss://finance@mypersonalfinance09.dfs.core.windows.net/silver/silver_transactions`

In [0]:
%sql
CREATE OR REPLACE VIEW gold_monthly_summary AS
SELECT
    DATE_TRUNC('month', date) AS month,
    
    SUM(CASE 
        WHEN Type = 'Income' THEN amount 
        ELSE 0 
    END) AS total_income,

    SUM(CASE 
        WHEN Type = 'Expenses' THEN amount 
        ELSE 0 
    END) AS total_expenses,

    SUM(CASE 
        WHEN Type = 'Income' THEN amount 
        ELSE -amount 
    END) AS net_cashflow,

    COUNT(*) AS total_transactions

FROM delta.`abfss://finance@mypersonalfinance09.dfs.core.windows.net/silver/silver_transactions`
GROUP BY 1
ORDER BY 1;

In [0]:
%sql
CREATE VIEW gold_category_spending AS
SELECT
    category,
    SUM(amount) AS total_spent,
    COUNT(*) AS transactions
FROM delta.`abfss://finance@mypersonalfinance09.dfs.core.windows.net/silver/silver_transactions`
WHERE Type = 'Expenses'
GROUP BY category
ORDER BY total_spent DESC;

In [0]:
%sql
CREATE VIEW gold_payment_mode_spending AS
SELECT
    payment_mode,
    SUM(amount) AS total_spent,
    COUNT(*) AS transactions
FROM delta.`abfss://finance@mypersonalfinance09.dfs.core.windows.net/silver/silver_transactions`
WHERE Type = 'Expenses'
GROUP BY payment_mode
ORDER BY total_spent DESC;

In [0]:
%sql
CREATE VIEW gold_daily_spending AS
SELECT
    date,
    SUM(amount) AS total_spent,
    COUNT(*) AS transactions
FROM delta.`abfss://finance@mypersonalfinance09.dfs.core.windows.net/silver/silver_transactions`
WHERE Type = 'Expenses'
GROUP BY date
ORDER BY date;

In [0]:
%sql
CREATE VIEW gold_top_expenses AS
SELECT
    date,
    category,
    payment_mode,
    amount,
    Notes
FROM delta.`abfss://finance@mypersonalfinance09.dfs.core.windows.net/silver/silver_transactions`
WHERE Type = 'Expenses'
ORDER BY amount DESC
LIMIT 20;

In [0]:
%sql
SELECT * FROM gold_top_expenses;

In [0]:
gold_base = "abfss://finance@mypersonalfinance09.dfs.core.windows.net/gold"

gold_views = [
    "gold_monthly_summary",
    "gold_category_spending",
    "gold_payment_mode_spending",
    "gold_daily_spending",
    "gold_top_expenses",
]

for view in gold_views:
    spark.sql(f"SELECT * FROM {view}") \
        .write.format("delta") \
        .mode("overwrite") \
        .save(f"{gold_base}/{view}")
    print(f"Saved {view} to {gold_base}/{view}")

print("All gold tables written successfully.")

In [0]:
%sql
CREATE or REPLACE VIEW gold_running_balance AS
SELECT
    transaction_id,
    date,
    Type,
    category,
    payment_mode,

    CASE
        WHEN Type = 'Income' THEN amount
        WHEN Type IN ('Expenses', 'Investment') THEN -amount
    END AS net_amount,

    SUM(
        CASE
            WHEN Type = 'Income' THEN amount
            WHEN Type IN ('Expenses', 'Investment') THEN -amount
        END
    ) OVER (
        ORDER BY date, transaction_id
    ) AS running_balance

FROM delta.`abfss://finance@mypersonalfinance09.dfs.core.windows.net/silver/silver_transactions`
ORDER BY date, transaction_id;

In [0]:
spark.sql(f"SELECT * FROM gold_running_balance") \
        .write.format("delta") \
        .mode("overwrite") \
        .save(f"{gold_base}/gold_running_balance")

In [0]:
%sql
SELECT * FROM gold_monthly_summary

In [0]:
%sql
SELECT
    month,
    total_income,
    total_expenses,
    net_cashflow
FROM gold_monthly_summary
ORDER BY month

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
category,
total_spent,

FROM gold_category_spending
ORDER BY total_spent DESC

In [0]:
%sql
SELECT
payment_mode,
total_spent
FROM gold_payment_mode_spending

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
date,
running_balance
FROM gold_running_balance
ORDER BY date

Databricks visualization. Run in Databricks to view.